# Electrode Map Generator
This interface automatically reads electrode layouts from DXF files and maps channels to PCB/headstage channels using the `AssemblyRouter`.

**Return**: \(3 \times N_{\text{channels}}\) matrix (X-location, Y-location, PCB/Headstage channel)

In [1]:
import csv
from pathlib import Path
import ezdxf
import ipywidgets as widgets
import numpy as np
from ipyfilechooser import FileChooser

import utils
from assembly_routing import AssemblyRouter

In [2]:
doc = ezdxf.readfile("../electrode_map_example/example1/50um_pitch_compatible_with_FPC.dxf")
#doc = ezdxf.readfile("../electrode_map_example/example2/Xiaomin_electrode_v2.dxf")
layers = sorted({e.dxf.layer for e in doc.modelspace()})
print(layers)

['L2D0_etching2', 'L3D0_etching3', 'electrode', 'etching', 'metalization', 'polyimide', 'polyimide_etch_borders']


In [3]:
# give the layer name of the electrode contours in the DXF file
# if you don't know, give None and instead the number of contours to extract (the largest ones will be used)
layer_name = "electrodes"
# if you don't know the layer name, give None and instead the number of contours to extract (the one closest to the origin will be used)
# ATTENtiON: we assume that the last contour is a hook loop, which we will remove from the electrode layout
num_contours = 64

In [4]:
dxf_chooser = FileChooser(filter_pattern="*.dxf")
probe_chooser = FileChooser(filter_pattern="*.json")
headstage_chooser = FileChooser(filter_pattern="*.json")
assembly_chooser = FileChooser(filter_pattern="*.csv")

display(widgets.HTML(
    "<h3>1. Electrode geometry</h3>"
    "<p>Select the electrode layout DXF file.</p>"
))
display(dxf_chooser)

display(widgets.HTML(
    "<h3>2. Probe mapping</h3>"
    "<p>Select the JSON mapping from electrode channels to connector pads.</p>"
))
display(probe_chooser)

display(widgets.HTML(
    "<h3>3. Headstage mapping</h3>"
    "<p>Select the JSON mapping from connector pads to headstage channels.</p>"
))
display(headstage_chooser)

display(widgets.HTML(
    "<h3>4. Assembly mapping</h3>"
    "<p>Select the CSV mapping from headstage channels to assembly channels.</p>"
))
display(assembly_chooser)

HTML(value='<h3>1. Electrode geometry</h3><p>Select the electrode layout DXF file.</p>')

FileChooser(path='/home/mathilda/work/ElectrodeToGeometry/electrode2geometry2', filename='', title='', show_hi…

HTML(value='<h3>2. Probe mapping</h3><p>Select the JSON mapping from electrode channels to connector pads.</p>…

FileChooser(path='/home/mathilda/work/ElectrodeToGeometry/electrode2geometry2', filename='', title='', show_hi…

HTML(value='<h3>3. Headstage mapping</h3><p>Select the JSON mapping from connector pads to headstage channels.…

FileChooser(path='/home/mathilda/work/ElectrodeToGeometry/electrode2geometry2', filename='', title='', show_hi…

HTML(value='<h3>4. Assembly mapping</h3><p>Select the CSV mapping from headstage channels to assembly channels…

FileChooser(path='/home/mathilda/work/ElectrodeToGeometry/electrode2geometry2', filename='', title='', show_hi…

In [5]:
run_button = widgets.Button(
    description="Generate Channel Map",
    button_style="success",
)
output = widgets.Output()

state = {
    "channel_map": None,
    "dxf_file": None,
}


def on_run_clicked(button):
    with output:
        output.clear_output(wait=True)

        if not dxf_chooser.value:
            print("Please select a DXF file.")
            return
        if not probe_chooser.value or not headstage_chooser.value:
            print("Please select both probe and headstage JSON files.")
            return
        if not assembly_chooser.value:
            print("Please select an assembly mapping CSV file.")
            return

        try:
            print("1. Loading electrode layout...")
            xy = utils.load_and_plot_electrodes(
                dxf_chooser.value,
                layer_name=layer_name,
                num_contours=num_contours,
            )

            print("2. Building Assembly Router & tracing paths...")
            router = AssemblyRouter()

            probe_json = Path(probe_chooser.value)
            headstage_json = Path(headstage_chooser.value)
            assembly_csv = Path(assembly_chooser.value)

            router.register_model_json(probe_json)
            router.register_model_json(headstage_json)

            probe_model = next(
                name for name, definition in router.models.items()
                if "headstage" not in definition.component_type.lower()
            )
            headstage_model = next(
                name for name, definition in router.models.items()
                if "headstage" in definition.component_type.lower()
            )

            probe_instance = "probe_0"
            headstage_instance = "headstage_0"

            router.add_instance(probe_instance, probe_model)
            router.add_instance(headstage_instance, headstage_model)

            router.load_assembly_csv(assembly_csv)

            probe_definition = router.models[probe_model]
            starts = [
                f"{probe_instance}.{channel}"
                for channel in probe_definition.mapping
            ]

            report = router.report(starts)
            complete_report = [
                item for item in report
                if item["status"] == "complete"
            ]

            if not complete_report:
                print("No complete signal paths discovered by Assembly Router.")
                return

            print("3. Generating channel map matrix...")
            num_electrodes = len(xy)
            channel_map = np.full((3, num_electrodes), np.nan)

            for item in complete_report:
                source_pin = item["source"].rsplit(".", 1)[1]
                destination_pin = item["destination"].rsplit(".", 1)[1]

                electrode_channel = int(source_pin)
                headstage_channel = int(destination_pin)

                electrode_index = electrode_channel - 1

                if 0 <= electrode_index < num_electrodes:
                    channel_map[0, electrode_index] = xy[electrode_index, 0]
                    channel_map[1, electrode_index] = xy[electrode_index, 1]
                    channel_map[2, electrode_index] = headstage_channel

            state["channel_map"] = channel_map
            state["dxf_file"] = dxf_chooser.value

            print("\nSuccess!")
            print(f"Shape: {channel_map.shape}")
            print(f"Sample column 0: {channel_map[:, 0]}")

        except Exception as error:
            print(f"Error generating channel map: {error}")


run_button.on_click(on_run_clicked)
display(run_button, output)

Button(button_style='success', description='Generate Channel Map', style=ButtonStyle())

Output()

In [6]:
export_button = widgets.Button(
    description="Export to CSV",
    button_style="info",
)
export_output = widgets.Output()


def on_export_clicked(button):
    with export_output:
        export_output.clear_output(wait=True)

        if state["channel_map"] is None:
            print("Generate a channel map first")
            return

        try:
            export_file = utils.export_channel_map_to_csv(
                state["channel_map"],
                state["dxf_file"],
            )
            print(f"Exported to: {export_file}")
        except Exception as error:
            print(f"Export failed: {error}")


export_button.on_click(on_export_clicked)
display(export_button, export_output)

Button(button_style='info', description='Export to CSV', style=ButtonStyle())

Output()